# Aula 4 — TF-IDF: quando frequência não basta

**Como destacar termos mais informativos em uma coleção de documentos**

Na Aula 3, aprendemos a transformar palavras em features usando Bag-of-Words e `CountVectorizer`.

Agora surge uma limitação importante: uma palavra muito frequente pode receber um valor alto simplesmente porque aparece em muitos documentos — mesmo que ajude pouco a diferenciá-los.

Nesta aula, vamos conhecer o **TF-IDF**, uma técnica clássica que combina frequência local e raridade global para produzir pesos mais informativos.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- explicar a intuição de TF, IDF e TF-IDF;
- comparar contagem simples e ponderação TF-IDF;
- usar `TfidfVectorizer` do `scikit-learn`;
- interpretar uma matriz TF-IDF;
- identificar termos comuns e termos discriminantes;
- compreender por que TF-IDF continua sendo um baseline importante em Text Intelligence.


## 📘 Glossário da aula

Conceitos centrais desta aula: **TF · IDF · TF-IDF · TfidfVectorizer · feature**.

Use o Glossário Vivo quando quiser revisar uma definição, conferir o termo técnico em inglês ou retomar a relação entre conceitos.

- [Glossário PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)
- [Glossary EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)

> O notebook explica o necessário para seguir a aula; o glossário serve para consolidar e aprofundar o vocabulário técnico.


## Antes de começar — trabalhe na sua própria cópia

Crie sua cópia do notebook no Kaggle antes de executar ou modificar qualquer célula.

```text
Notebook oficial = referência do curso
Cópia do aluno    = ambiente pessoal de aprendizagem
```


## 2. O problema com a contagem simples

Considere três mensagens:

```text
cliente satisfeito com atendimento rápido
cliente satisfeito com atendimento eficiente
cliente insatisfeito com atendimento demorado
```

Termos como `cliente` e `atendimento` aparecem em todos os documentos.

Eles são frequentes, mas ajudam pouco a distinguir uma mensagem da outra.

Já termos como `rápido`, `eficiente`, `insatisfeito` e `demorado` aparecem em menos documentos e podem carregar mais informação discriminante.

**Pergunta central:** como reduzir o peso de termos muito comuns e aumentar o peso relativo dos termos mais específicos?


## 3. A ideia do TF-IDF

TF-IDF combina duas ideias:

### TF — Term Frequency

Mede quanto um termo aparece em um documento.

### IDF — Inverse Document Frequency

Reduz o peso de termos que aparecem em muitos documentos e aumenta o peso relativo dos termos mais raros no corpus.

### TF-IDF

Combina os dois componentes para atribuir um peso a cada termo em cada documento.

A intuição é:

```text
muito frequente em um documento
+ relativamente raro no corpus
→ termo potencialmente mais informativo
```


## 4. Primeiro, observe a contagem simples

Vamos começar com `CountVectorizer` para construir uma referência.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

texts = [
    "cliente satisfeito com atendimento rápido",
    "cliente satisfeito com atendimento eficiente",
    "cliente insatisfeito com atendimento demorado",
]

count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(texts)

count_features = count_vectorizer.get_feature_names_out()

count_df = pd.DataFrame(
    X_count.toarray(),
    columns=count_features,
    index=["doc_1", "doc_2", "doc_3"],
)

count_df


### O que observar

`cliente`, `com` e `atendimento` aparecem em todos os documentos.

Na matriz de contagem, porém, eles recebem o mesmo tipo de valor que termos mais específicos.

**Checkpoint 1:** identifique quais colunas ajudam pouco a diferenciar os documentos.


## 5. Agora aplique TF-IDF

Vamos usar `TfidfVectorizer`, também do `scikit-learn`.

A API é muito parecida com `CountVectorizer`, o que facilita a comparação.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(texts)

tfidf_features = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_features,
    index=["doc_1", "doc_2", "doc_3"],
)

tfidf_df.round(3)


### O que interpretar

Agora os valores deixam de ser contagens inteiras e passam a representar **pesos**.

Termos presentes em todos os documentos tendem a receber menor peso relativo.

Termos menos comuns, quando aparecem em um documento, tendem a receber peso maior.

**Checkpoint 2:** compare os pesos de `atendimento` com os de `rápido`, `eficiente` e `demorado`.


## 6. Inspecionando o IDF

`TfidfVectorizer` também permite observar o componente IDF aprendido para cada termo.

Isso ajuda a enxergar quais palavras são consideradas mais ou menos raras no corpus.


In [ ]:
idf_df = pd.DataFrame({
    "term": tfidf_features,
    "idf": tfidf_vectorizer.idf_,
}).sort_values("idf", ascending=False)

idf_df.reset_index(drop=True)


### O que observar

Quanto maior o IDF, menos frequente o termo é no conjunto de documentos.

Termos presentes em todos os documentos recebem IDF menor.

Isso não significa automaticamente que um termo raro seja semanticamente importante. TF-IDF mede **padrão estatístico de distribuição**, não significado.


## 7. TF-IDF como Feature Engineering para texto

Na Aula 3, cada termo virou uma feature de contagem.

Agora continuamos usando uma feature por termo, mas alteramos a forma de medir sua importância.

```text
Bag-of-Words
→ frequência bruta

TF-IDF
→ frequência ponderada pela raridade no corpus
```

Essa transformação é um exemplo direto de **engenharia de features aplicada a texto**.


## 8. Quando TF-IDF é útil?

TF-IDF costuma funcionar muito bem como baseline em problemas como:

- classificação de documentos;
- categorização de mensagens;
- recuperação de informação;
- busca por similaridade lexical;
- identificação de termos discriminantes.

Sua força está na combinação de:

- simplicidade;
- velocidade;
- baixo custo computacional;
- interpretabilidade.


## 9. Limitações

TF-IDF ainda possui limitações importantes:

- não compreende significado semântico profundo;
- não representa diretamente ordem completa das palavras;
- sinônimos continuam sendo features diferentes;
- palavras iguais em contextos diferentes continuam compartilhando a mesma dimensão.

Essas limitações prepararão o terreno para representações mais avançadas em aulas futuras.


## 10. Exercício guiado

Use as mensagens abaixo:

```python
messages = [
    "produto excelente entrega rápida",
    "produto excelente qualidade ótima",
    "produto ruim entrega atrasada",
]
```

Seu código deve:

1. criar um `TfidfVectorizer`;
2. ajustar e transformar as mensagens;
3. obter os nomes das features;
4. construir um `DataFrame` com os pesos TF-IDF;
5. arredondar a visualização para três casas decimais;
6. identificar visualmente quais termos parecem mais discriminantes.


In [ ]:
# Escreva sua solução aqui.

messages = [
    "produto excelente entrega rápida",
    "produto excelente qualidade ótima",
    "produto ruim entrega atrasada",
]

# Continue a partir daqui.


### Dica e solução

Execute **uma vez** a próxima célula para preparar `q4.hint()` e `q4.solution()`.

A dica indica as bibliotecas, classes e métodos relevantes para construir a resposta.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q4 = TILExercise(
    hint_text=(
        "Use **scikit-learn + pandas**. "
        "A classe principal é `sklearn.feature_extraction.text.TfidfVectorizer`. "
        "Os métodos mais úteis são `fit_transform()` e `get_feature_names_out()`. "
        "Para inspecionar os valores, construa um `pd.DataFrame(...)` com `X.toarray()` e use `.round(3)`."
    ),
    solution_text=(
        "Uma possível solução executável é:\n\n"
        "```python\n"
        "from sklearn.feature_extraction.text import TfidfVectorizer\n"
        "import pandas as pd\n\n"
        "messages = [\n"
        "    'produto excelente entrega rápida',\n"
        "    'produto excelente qualidade ótima',\n"
        "    'produto ruim entrega atrasada',\n"
        "]\n\n"
        "vectorizer = TfidfVectorizer()\n"
        "X = vectorizer.fit_transform(messages)\n"
        "features = vectorizer.get_feature_names_out()\n\n"
        "result = pd.DataFrame(X.toarray(), columns=features)\n"
        "display(result.round(3))\n"
        "```\n\n"
        "Observe que termos compartilhados por vários documentos tendem a receber menor peso relativo do que termos mais específicos."
    ),
)

print("Exercício preparado. Tente resolver antes de usar q4.hint() ou q4.solution().")


In [ ]:
# Remova o # da linha abaixo se quiser uma dica.
# q4.hint()


In [ ]:
# Remova o # da linha abaixo para revelar a solução.
# q4.solution()


## 11. Reprodutibilidade

Esta aula utiliza apenas dados definidos no próprio notebook.

- linguagem: Python;
- bibliotecas: `scikit-learn` e `pandas`;
- representação: `TfidfVectorizer`;
- acelerador: CPU;
- internet: desabilitada;
- dataset externo: nenhum.


## 12. Resumo

Nesta aula, você aprendeu que:

- contagem simples pode supervalorizar termos comuns;
- TF mede frequência dentro do documento;
- IDF reduz o peso de termos comuns no corpus;
- TF-IDF combina essas duas ideias;
- `TfidfVectorizer` transforma documentos em features ponderadas;
- TF-IDF é simples, rápido, interpretável e útil como baseline;
- TF-IDF mede distribuição estatística, não significado profundo.

### Ideia principal

```text
Nem toda palavra frequente é informativa.
TF-IDF ajuda a destacar termos que diferenciam documentos.
```

**Fim da Aula 4.**
